# Etapa 3. O MZA-FAO em um município e um ano

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/03_mza_um_ciclo.ipynb)

Este notebook simula **um ciclo** do milho (Santa Helena, semeadura em 05/02) e mostra o que o modelo faz, dia a
dia:

1. soma os **graus-dia** (Tb = 10 °C) até 1.450 °C dia, a maturação de um híbrido precoce;
2. divide o ciclo nas **cinco fases** de Doorenbos & Kassam: estabelecimento, vegetativo, floração, enchimento de
   grãos e maturação;
3. calcula a **ETc = Kc × ETo**; a raiz cresce de 0,10 m até 0,40 m no fim do vegetativo;
4. faz o **balanço hídrico da cultura** e obtém a **ETR**, a água que a planta de fato transpirou;
5. calcula a **produtividade potencial (PP)** com a radiação e a temperatura;
6. penaliza a PP pelo déficit de cada fase com o **ky** e obtém a **produtividade atingível (PA)**.

As funções estão em `src/mza_oeste.py`. Vale abrir esse arquivo e ler: está comentado.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import mza_oeste as M          # as funções do projeto (pasta src/)

clima = pd.read_parquet("dados/clima_oeste.parquet")
mun = pd.read_csv("dados/municipios_oeste.csv").set_index("municipio")

sh = mun.loc["Santa Helena"]
clima_sh = clima[clima["cod_ibge"] == sh["cod_ibge"]].set_index("data")
print(f"Santa Helena: lat {sh['latitude']:.2f}, CAD {sh['dta_mm_m']:.0f} mm/m")

## 1. Um ciclo

`M.simular` devolve um **dicionário** com a PP (`PPf`), a PA, a duração do ciclo e, para cada fase, a chuva, o
déficit, as temperaturas e o **ISNA** (índice de satisfação das necessidades de água, ISNA = ETR/ETc; 1 = sem
falta de água). Com `diario=True`, devolve também a tabela diária do ciclo.

In [ ]:
r, dia = M.simular(clima_sh, lat=sh["latitude"], dta=sh["dta_mm_m"], semeadura="2019-02-05", diario=True)

print(f"Ciclo: {r['ciclo_dias']} dias; floração em {r['data_floracao']:%d/%m}; maturação em {r['data_maturacao']:%d/%m}")
print(f"PP = {r['PPf']:.0f} kg/ha    PA = {r['PA']:.0f} kg/ha    PA/PP = {r['PA'] / r['PPf']:.2f}")
pd.DataFrame({f: {"chuva (mm)": r[f"chuva_{f}"], "ETc (mm)": r[f"ETc_{f}"], "ETR (mm)": r[f"ETR_{f}"],
                  "ISNA": r[f"ISNA_{f}"]} for f in M.FASES}).round(2)

## 2. O ciclo em gráfico

Barras = chuva; linhas = ETc (o que a planta precisaria) e ETR (o que ela conseguiu). Quando a ETR fica abaixo da
ETc, há **déficit**. As faixas coloridas são as fases.

In [ ]:
def grafico_ciclo(dia, titulo):
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(dia.index, dia["pr"], color="lightsteelblue", label="Chuva")
    ax.plot(dia.index, dia["ETc"], color="black", label="ETc")
    ax.plot(dia.index, dia["ETR"], color="tab:red", label="ETR")
    ax.set_ylim(0, 25)                                         # eixo fixo, para comparar anos
    cores = dict(zip(M.FASES, ["#f0f0f0", "#e0f3db", "#fde0dd", "#fff7bc", "#efedf5"]))
    nomes = dict(zip(M.FASES, ["estabelec.", "vegetativo", "floração", "enchimento", "maturação"]))
    for fase, g in dia.groupby("fase_ky", sort=False):        # uma faixa colorida por fase
        ax.axvspan(g.index[0], g.index[-1], color=cores[fase], alpha=0.6, zorder=0)
        ax.text(g.index[len(g) // 2], 23.5, nomes[fase], ha="center", fontsize=8)
    ax.set_ylabel("mm/dia")
    ax.set_title(titulo)
    ax.legend(loc="upper right", bbox_to_anchor=(1, 0.9))
    plt.show()

grafico_ciclo(dia, "Santa Helena, semeadura em 05/02/2019")

## 3. A equação do ky, na mão

A PA é obtida pela forma do **produtório** (Battisti et al., 2013): fase por fase, a produtividade é reduzida pela
fração perdida naquela fase.

$$PA = PP \times \prod_{i=1}^{5} \left[1 - k_{y,i}\left(1 - \frac{ETR_i}{ETc_i}\right)\right]$$

Este bloco é o coração do trabalho. **Note:** conhecidos o ISNA de cada fase e a PP, a PA sai de uma conta simples
para **qualquer** conjunto de ky. Por isso a calibração da Etapa 6 não precisa rodar o balanço hídrico de novo.

In [ ]:
fator = 1.0
for f in M.FASES:
    perda = M.KY_DK[f] * (1 - r[f"ISNA_{f}"])        # fração da produtividade perdida na fase
    fator = fator * max(1 - perda, 0)                # nunca negativo
    print(f"{f:16s} ISNA = {r[f'ISNA_{f}']:.2f}   perda = {perda:.2f}   fator acumulado = {fator:.3f}")
print(f"PA recalculada = {r['PPf'] * fator:.0f} kg/ha (a função deu {r['PA']:.0f})")

## 4. Um ano seco: 2020

Segundo o IBGE, Santa Helena colheu 5.800 kg/ha em 2019 e só 2.200 kg/ha em 2020, ano de estiagem forte no
extremo oeste. Compare os ISNA dos dois anos.

In [ ]:
r20, dia20 = M.simular(clima_sh, sh["latitude"], sh["dta_mm_m"], "2020-02-05", diario=True)
grafico_ciclo(dia20, "Santa Helena, semeadura em 05/02/2020")
pd.DataFrame({"2019": {f: r[f"ISNA_{f}"] for f in M.FASES}, "2020": {f: r20[f"ISNA_{f}"] for f in M.FASES}}).round(2)

In [ ]:
print(f"2019: PP = {r['PPf']:.0f}  PA = {r['PA']:.0f} kg/ha")
print(f"2020: PP = {r20['PPf']:.0f}  PA = {r20['PA']:.0f} kg/ha")

### Ponto para discussão com o orientador

Em 2020 a PA simulada fica perto de **zero**, mas o IBGE registrou cerca de 2.200 kg/ha em Santa Helena. Duas
razões possíveis, que serão investigadas nas próximas etapas:

- o **ky da floração (1,5)** combinado com o produtório pune demais: um ISNA de 0,35 nessa fase já elimina quase
  toda a produção (calcule: 1 − 1,5 × 0,65);
- o clima é o da **sede** do município, e a chuva varia muito dentro dele em anos de estiagem.

Note também a **PP**: perto de 6.000–7.000 kg/ha. Em anos bons o IBGE registra 6.000 kg/ha ou mais em Santa
Helena. Uma PP tão próxima do real sugere que ela está **subestimada**, o que será verificado na Etapa 5.

### Exercício

1. Simule 2019 para as semeaduras de 15/01 e de 15/03. Como mudam a duração do ciclo, a PP e a PA? Por quê?
2. Refaça o produtório com ky da floração = 0,8. Quanto muda a PA de 2020?

### O que escrever

A subseção **Modelo da Zona Agroecológica da FAO** do Material e Métodos: as equações numeradas da fenologia
(graus-dia), do balanço hídrico, da PP e da PA, com a legenda de cada variável logo abaixo.

In [ ]:
# escreva sua resposta aqui